<i18n value="3a66b3fc-6b92-436c-8b22-95b91bddeac3"/>

# 델타 테이블에 쓰기
델타 레이크 테이블은 클라우드 객체 스토리지의 데이터 파일로 지원되는 테이블에 ACID 호환 업데이트를 제공합니다.

이 노트북에서는 델타 레이크에서 업데이트를 처리하는 SQL 구문을 살펴보겠습니다. 많은 작업이 표준 SQL이지만, Spark 및 델타 레이크 실행에 맞게 약간의 변형이 있습니다.

## 학습 목표
이 과정을 마치면 다음을 수행할 수 있어야 합니다.
- **`INSERT OVERWRITE`**를 사용하여 데이터 테이블 덮어쓰기
- **`INSERT INTO`**를 사용하여 테이블에 추가
- **`MERGE INTO`**를 사용하여 테이블에 추가, 업데이트 및 삭제
- **`COPY INTO`**를 사용하여 테이블에 데이터를 증분 방식으로 수집

<i18n value="3fa1b6f1-9faa-453f-b033-c5f2cf011703"/>

## 설치 실행

설치 스크립트는 이 노트북의 나머지 부분을 실행하는 데 필요한 데이터를 생성하고 값을 선언합니다.

In [0]:
%run ../Includes/Classroom-Setup-04.4

Python interpreter will be restarted.
Python interpreter will be restarted.


Resetting the learning environment:
| No action taken

Skipping install of existing datasets to "dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02"

Validating the locally installed datasets:
| listing local files...(3 seconds)
| validation completed...(3 seconds total)

Creating & using the schema "3dt005_nuk5_da_dewd" in the catalog "hive_metastore"...(1 seconds)

Cloning the sales table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/sales_hist...(5 seconds / 10,510 records)
Cloning the users table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/users_hist...(5 seconds / 251,501 records)
Cloning the events table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/events_hist...(5 seconds / 485,696 records)
Cloning the users_update table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/users_update...(5 seconds / 917 re

<i18n value="3a8cba57-5d9e-4514-b589-d263a2f02f74"/>

## 완전 덮어쓰기

덮어쓰기를 사용하면 테이블의 모든 데이터를 원자적으로 바꿀 수 있습니다. 테이블을 삭제하고 다시 생성하는 대신 테이블을 덮어쓰면 여러 가지 이점이 있습니다.
- 테이블 덮어쓰기는 디렉터리를 재귀적으로 나열하거나 파일을 삭제할 필요가 없으므로 훨씬 빠릅니다.
- 테이블의 이전 버전이 여전히 존재하므로 시간 여행을 사용하여 이전 데이터를 쉽게 검색할 수 있습니다.
- 원자적 연산입니다. 테이블을 삭제하는 동안에도 동시 쿼리에서 테이블을 읽을 수 있습니다.
- ACID 트랜잭션 보장으로 인해 테이블 ​​덮어쓰기가 실패하더라도 테이블은 이전 상태로 유지됩니다.

Spark SQL은 완전 덮어쓰기를 수행하는 두 가지 간단한 방법을 제공합니다.

일부 학생들은 이전 CTAS 문 수업에서 실제로 CRAS 문을 사용했다는 것을 알아차렸을 것입니다(셀을 여러 번 실행할 경우 발생할 수 있는 오류를 방지하기 위해).

**`CREATE OR REPLACE TABLE`**(CRAS) 문은 실행될 때마다 테이블의 내용을 완전히 바꿉니다.

In [0]:
CREATE OR REPLACE TABLE events AS
SELECT * FROM parquet.`${da.paths.datasets}/ecommerce/raw/events-historical`

num_affected_rows,num_inserted_rows


<i18n value="a4090f92-557e-4d23-a808-155356270d1b"/>

테이블 기록을 검토해 보니 이 테이블의 이전 버전이 대체된 것으로 나타났습니다.

In [0]:
DESCRIBE HISTORY events

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-03-20T06:02:29.000+0000,148112838596515,3dt005@msacademy.msai.kr,CREATE OR REPLACE TABLE AS SELECT,"Map(isManaged -> true, description -> null, partitionBy -> [], properties -> {})",null,List(4480172884687505),0319-023558-6tyoy5l4,0,WriteSerializable,false,"Map(numFiles -> 4, numOutputRows -> 485696, numOutputBytes -> 15283922)",null,Databricks-Runtime/11.3.x-scala2.12
0,2026-03-20T06:02:11.000+0000,148112838596515,3dt005@msacademy.msai.kr,CLONE,"Map(source -> delta.`dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/events_hist`, sourceVersion -> 1, isShallow -> true)",null,List(4480172884687505),0319-023558-6tyoy5l4,-1,Serializable,false,"Map(removedFilesSize -> 0, numRemovedFiles -> 0, sourceTableSize -> 14998176, numCopiedFiles -> 0, copiedFilesSize -> 0, sourceNumOfFiles -> 1)",null,Databricks-Runtime/11.3.x-scala2.12


<i18n value="06628fba-431d-4cb7-bd52-449fdc203cb2"/>

**`INSERT OVERWRITE`**는 위와 거의 동일한 결과를 제공합니다. 대상 테이블의 데이터가 쿼리의 데이터로 대체됩니다.

**`INSERT OVERWRITE`**:

- 기존 테이블만 덮어쓸 수 있으며, CRAS 문처럼 새 테이블을 생성할 수 없습니다.
- 현재 테이블 스키마와 일치하는 새 레코드로만 덮어쓸 수 있습니다. 따라서 다운스트림 소비자를 방해하지 않고 기존 테이블을 덮어쓸 수 있는 "더 안전한" 기법입니다.
- 개별 파티션을 덮어쓸 수 있습니다.

In [0]:
INSERT OVERWRITE sales
SELECT * FROM parquet.`${da.paths.datasets}/ecommerce/raw/sales-historical/`

num_affected_rows,num_inserted_rows
10510,10510


<i18n value="c668da22-a355-441f-aeab-a513997b8d71"/>

CRAS 명령문과는 다른 지표가 표시되며, 테이블 기록에도 작업이 다르게 기록됩니다.

In [0]:
DESCRIBE HISTORY sales

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-03-20T06:02:38.000+0000,148112838596515,3dt005@msacademy.msai.kr,WRITE,"Map(mode -> Overwrite, partitionBy -> [])",null,List(4480172884687505),0319-023558-6tyoy5l4,0,WriteSerializable,false,"Map(numFiles -> 4, numOutputRows -> 10510, numOutputBytes -> 353497)",null,Databricks-Runtime/11.3.x-scala2.12
0,2026-03-20T06:02:00.000+0000,148112838596515,3dt005@msacademy.msai.kr,CLONE,"Map(source -> delta.`dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/sales_hist`, sourceVersion -> 1, isShallow -> true)",null,List(4480172884687505),0319-023558-6tyoy5l4,-1,Serializable,false,"Map(removedFilesSize -> 0, numRemovedFiles -> 0, sourceTableSize -> 334175, numCopiedFiles -> 0, copiedFilesSize -> 0, sourceNumOfFiles -> 1)",null,Databricks-Runtime/11.3.x-scala2.12


<i18n value="4fa20011-e560-45d5-85c0-c82930974804"/>

여기서 가장 큰 차이점은 Delta Lake가 스키마 온 쓰기(schema on write)를 적용하는 방식과 관련이 있습니다.

CRAS 문을 사용하면 대상 테이블의 내용을 완전히 재정의할 수 있지만, 스키마를 변경하려고 하면 (선택적인 설정을 제공하지 않는 한) **`INSERT OVERWRITE`**가 실패합니다.

아래 셀의 주석 처리를 제거하고 실행하면 예상 오류 메시지가 생성됩니다.

In [0]:
-- INSERT OVERWRITE sales
-- SELECT *, current_timestamp() FROM parquet.`${da.paths.datasets}/ecommerce/raw/sales-historical`

<i18n value="02c9c3f2-2996-42b5-9d56-60967a0031c0"/>

## 행 추가

**`INSERT INTO`**를 사용하여 기존 델타 테이블에 새로운 행을 원자적으로 추가할 수 있습니다. 이렇게 하면 기존 테이블에 증분 업데이트를 적용할 수 있으므로 매번 덮어쓰는 것보다 훨씬 효율적입니다.

**`INSERT INTO`**를 사용하여 **`sales`** 테이블에 새로운 판매 레코드를 추가합니다.

In [0]:
INSERT INTO sales
SELECT * FROM parquet.`${da.paths.datasets}/ecommerce/raw/sales-30m`

num_affected_rows,num_inserted_rows
29,29


<i18n value="3a0a94f8-43fa-4dc3-be0c-110b07f1db89"/>

**`INSERT INTO`**에는 동일한 레코드를 여러 번 삽입하는 것을 방지하는 기본 제공 기능이 없습니다. 위 셀을 다시 실행하면 대상 테이블에 동일한 레코드가 기록되어 중복 레코드가 생성됩니다.

<i18n value="cad7b32f-3e35-4962-a911-823d92f2f653"/>

## 업데이트 병합

**`MERGE`** SQL 작업을 사용하여 원본 테이블, 뷰 또는 DataFrame의 데이터를 대상 Delta 테이블에 업서트할 수 있습니다. Delta Lake는 **`MERGE`**에서 삽입, 업데이트 및 삭제를 지원하며, 고급 사용 사례를 지원하기 위해 SQL 표준을 뛰어넘는 확장된 구문을 지원합니다.

<strong><code>
MERGE INTO target a<br/>
USING source b<br/>
ON {merge_condition}<br/>
WHEN MATCHED THEN {matched_action}<br/>
WHEN NOT MATCHED THEN {not_matched_action}<br/>
</code></strong>

**`MERGE`** 작업을 사용하여 업데이트된 이메일과 새로운 사용자를 포함하여 이전 사용자 데이터를 업데이트합니다.

In [0]:
CREATE OR REPLACE TEMP VIEW users_update AS 
SELECT *, current_timestamp() AS updated 
FROM parquet.`${da.paths.datasets}/ecommerce/raw/users-30m`

<i18n value="edc2f42c-8cd2-47c6-a8e6-09b1541e1e00"/>

**`MERGE`**의 주요 이점:
* 업데이트, 삽입 및 삭제가 단일 트랜잭션으로 완료됩니다.
* 일치하는 필드 외에도 여러 조건을 추가할 수 있습니다.
* 사용자 지정 로직을 구현하기 위한 광범위한 옵션을 제공합니다.

아래에서는 현재 행에 **`NULL`** 이메일이 있고 새 행에는 없는 경우에만 레코드를 업데이트합니다.

새 배치에서 일치하지 않는 모든 레코드가 삽입됩니다.

In [0]:
MERGE INTO users a
USING users_update b
ON a.user_id = b.user_id
WHEN MATCHED AND a.email IS NULL AND b.email IS NOT NULL THEN
  UPDATE SET email = b.email, updated = b.updated
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
983,72,0,911


<i18n value="90e330ce-5596-4fa8-a474-78442e48ae67"/>

**`MATCHED`** 및 **`NOT MATCHED`** 조건 모두에 대해 이 함수의 동작을 명시적으로 지정합니다. 여기에 제시된 예는 적용 가능한 논리의 예일 뿐이며, 모든 **`MERGE`** 동작을 나타내는 것은 아닙니다.

<i18n value="493c7f22-4011-4b08-ad79-4b2a4ef2a12f"/>

## 중복 제거를 위한 삽입 전용 병합

일반적인 ETL 사용 사례는 일련의 추가 작업을 통해 로그 또는 기타 모든 추가 데이터 세트를 델타 테이블에 수집하는 것입니다.

많은 소스 시스템에서 중복 레코드가 생성될 수 있습니다. 병합을 사용하면 삽입 전용 병합을 수행하여 중복 레코드 삽입을 방지할 수 있습니다.

이 최적화된 명령은 동일한 **`MERGE`** 구문을 사용하지만 **`WHEN NOT MATCHED`** 절만 제공합니다.

아래에서는 이 절을 사용하여 동일한 **`user_id`** 및 **`event_timestamp`**를 가진 레코드가 **`events`** 테이블에 이미 있는지 확인합니다.

In [0]:
MERGE INTO events a
USING events_update b
ON a.user_id = b.user_id AND a.event_timestamp = b.event_timestamp
WHEN NOT MATCHED AND b.traffic_source = 'email' THEN 
  INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
128,0,0,128


<i18n value="1162da2b-3c56-42ff-b6b8-2532276b03cd"/>

## 증분 로드

**`COPY INTO`**는 SQL 엔지니어에게 외부 시스템에서 데이터를 증분 방식으로 수집할 수 있는 멱등 옵션을 제공합니다.

이 작업에는 몇 가지 예상 사항이 있습니다.
- 데이터 스키마는 일관성이 있어야 합니다.
- 중복 레코드는 제외되거나 다운스트림에서 처리되어야 합니다.

이 작업은 예측 가능하게 증가하는 데이터의 경우 전체 테이블 스캔보다 훨씬 저렴할 수 있습니다.

여기서는 정적 디렉터리에서 간단한 실행을 보여드리지만, 진정한 가치는 시간이 지남에 따라 여러 번 실행하여 소스에서 새 파일을 자동으로 가져오는 것입니다.

In [0]:
COPY INTO sales
FROM "${da.paths.datasets}/ecommerce/raw/sales-30m"
FILEFORMAT = PARQUET

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
29,29,0


<i18n value="567f82f1-956b-42c5-98d7-7914675b105e"/>

이 레슨과 관련된 표와 파일을 삭제하려면 다음 셀을 실행하세요.

In [0]:
%python
DA.cleanup()

Resetting the learning environment:
| dropping the schema "3dt005_nuk5_da_dewd"...(3 seconds)
| removing the working directory "dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineering-with-databricks"...(0 seconds)

Validating the locally installed datasets:
| listing local files...(3 seconds)
| validation completed...(3 seconds total)
